In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260615_154932"
snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))
snapshots

,ts,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,...,ask_delta,quote_churn,future_mid_100ms,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms
0,1781516013114,BTCUSDT,65606.06,65606.07,65606.065,65606.064810,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65606.065,0.000000,65614.875,0.000134,65609.215,0.000048
1,1781516013214,BTCUSDT,65606.06,65606.07,65606.065,65606.064869,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65606.065,0.000000,65614.875,0.000134,65606.685,0.000009
2,1781516013314,BTCUSDT,65606.06,65606.07,65606.065,65606.064708,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65606.065,0.000000,65614.875,0.000134,65606.685,0.000009
3,1781516013414,BTCUSDT,65606.06,65606.07,65606.065,65606.064708,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65609.355,0.000050,65614.875,0.000134,65606.685,0.000009
4,1781516013514,BTCUSDT,65606.06,65606.07,65606.065,65606.064860,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65614.875,0.000134,65614.875,0.000134,65606.685,0.000009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225582,1781538571914,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN
225583,1781538572014,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN
225584,1781538572114,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN
225585,1781538572214,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
"""
Key Research Questions

This project is designed to investigate:

Which regimes favor passive liquidity provision? medium frequency trends (1000ms rolling window)

"""

# STEP 1 — Load raw data
df = snapshots
df["ts"] = pd.to_datetime(df["ts"], unit="ms")
df = df.set_index("ts")

# STEP 2 — Build REGIME FEATURES (ONLY past info) slower trends - 1000ms

"""
2. Choose regime window (critical design choice)
Start simple:
"""

feature_cols = [
    "volatility",
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "quote_churn",
    "inventory",
    "inventory_vol",
    "microprice_error"
]

WINDOW = "1s"   # later try 2s, 5s

regime_df = pd.DataFrame()

regime_df["volatility"] = df["mid"].pct_change().rolling(WINDOW).std()
regime_df["spread"] = df["spread"].rolling(WINDOW).mean()
regime_df["order_imbalance"] = df["order_imbalance"].rolling(WINDOW).mean()
regime_df["trade_imbalance"] = df["trade_imbalance"].rolling(WINDOW).mean()
regime_df["quote_churn"] = df["quote_churn"].rolling(WINDOW).mean()
regime_df["inventory"] = df["inventory"].rolling(WINDOW).mean()
regime_df["inventory_vol"] = df["inventory"].rolling(WINDOW).std()
regime_df["microprice_error"] = (df["mid"] - df["microprice"]).rolling(WINDOW).mean()

regime_df = regime_df.dropna()
regime_df

,volatility,spread,order_imbalance,trade_imbalance,quote_churn,inventory,inventory_vol,microprice_error
ts,,,,,,,,
2026-06-15 09:33:33.314,0.0,0.01,-0.038463,-0.874730,0.0,0.000000,0.000000,0.000204
2026-06-15 09:33:33.414,0.0,0.01,-0.042843,-0.879204,0.0,0.000000,0.000000,0.000226
2026-06-15 09:33:33.514,0.0,0.01,-0.039408,-0.881888,0.0,0.000000,0.000000,0.000209
2026-06-15 09:33:33.614,0.0,0.01,0.034790,-0.883678,0.0,0.000000,0.000000,-0.000162
2026-06-15 09:33:33.714,0.0,0.01,0.087847,-0.884956,0.0,0.000000,0.000000,-0.000427
...,...,...,...,...,...,...,...,...
2026-06-15 15:49:31.914,0.0,0.01,0.309038,-0.424231,0.0,-0.421456,0.000388,-0.001533
2026-06-15 15:49:32.014,0.0,0.01,0.350889,-0.386807,0.0,-0.421362,0.000442,-0.001741
2026-06-15 15:49:32.114,0.0,0.01,0.392706,-0.349383,0.0,-0.421268,0.000470,-0.001950


In [3]:
# STEP 3 — Train regime model
scaler = StandardScaler()

X = regime_df[feature_cols].values
X_scaled = scaler.fit_transform(X)

n_regimes = 3  # start small: 2–5 max

model = GaussianMixture(
    n_components=n_regimes,
    covariance_type="full",
    random_state=42
)

regime_df["regime"] = model.fit_predict(X_scaled)

In [4]:
eval_df = df.copy()

times = eval_df.index          # DatetimeIndex
mid = eval_df["mid"].values

horizon_ms = 1000
HORIZON = pd.Timedelta(milliseconds=horizon_ms)

future_return = np.full(len(df), np.nan)
future_volatility = np.full(len(df), np.nan)
future_direction = np.full(len(df), np.nan)

for i in range(len(df)):

    target_time = times[i] + HORIZON

    # first observation at or after t + 1000ms
    j = times.searchsorted(target_time)

    if j >= len(df):
        continue

    p0 = mid[i]
    p1 = mid[j]

    # future window [i, j]
    window = mid[i:j+1]

    # Need at least 2 observations
    if len(window) < 2:
        continue

    # 1. Future return
    future_return[i] = (p1 - p0) / p0

    # 2. Realized volatility over next 1000ms
    returns = np.diff(window) / window[:-1]
    future_volatility[i] = np.std(returns)

    # 3. Future direction
    # If result ≈ +1
    # almost always up moves after this regime
    # strong bullish bias
    # If result ≈ -1
    # almost always down moves after this regime
    # bearish bias
    # If result ≈ 0
    # no directional bias
    # pure noise / mean reversion / stable
    future_direction[i] = np.sign(p1 - p0)

eval_df["future_return"] = future_return
eval_df["future_volatility"] = future_volatility
eval_df["future_direction"] = future_direction

eval_df = eval_df.dropna(subset=[ "future_return", "future_volatility", "future_direction"])
eval_df

,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,order_imbalance,...,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms,future_return,future_volatility,future_direction
ts,,,,,,,,,,,,,,,,,,,,,
2026-06-15 09:33:33.114,BTCUSDT,65606.06,65606.07,65606.065,65606.064810,6560606,6560607,6560606,0.01,-0.035510,...,0.0,65606.065,0.000000,65614.875,0.000134,65609.215,0.000048,0.000134,0.000028,1.0
2026-06-15 09:33:33.214,BTCUSDT,65606.06,65606.07,65606.065,65606.064869,6560606,6560607,6560606,0.01,-0.023879,...,0.0,65606.065,0.000000,65614.875,0.000134,65606.685,0.000009,0.000134,0.000028,1.0
2026-06-15 09:33:33.314,BTCUSDT,65606.06,65606.07,65606.065,65606.064708,6560606,6560607,6560606,0.01,-0.055999,...,0.0,65606.065,0.000000,65614.875,0.000134,65606.685,0.000009,0.000134,0.000028,1.0
2026-06-15 09:33:33.414,BTCUSDT,65606.06,65606.07,65606.065,65606.064708,6560606,6560607,6560606,0.01,-0.055984,...,0.0,65609.355,0.000050,65614.875,0.000134,65606.685,0.000009,0.000134,0.000028,1.0
2026-06-15 09:33:33.514,BTCUSDT,65606.06,65606.07,65606.065,65606.064860,6560606,6560607,6560606,0.01,-0.025666,...,0.0,65614.875,0.000134,65614.875,0.000134,65606.685,0.000009,0.000134,0.000028,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-15 15:49:30.914,BTCUSDT,67161.98,67161.99,67161.985,67161.986500,6716198,6716199,6716198,0.01,0.302432,...,0.0,67161.985,0.000000,67161.985,0.000000,NaN,NaN,0.000000,0.000000,0.0
2026-06-15 15:49:31.014,BTCUSDT,67161.98,67161.99,67161.985,67161.985536,6716198,6716199,6716198,0.01,0.109397,...,0.0,67161.985,0.000000,67161.985,0.000000,NaN,NaN,0.000000,0.000000,0.0
2026-06-15 15:49:31.114,BTCUSDT,67161.98,67161.99,67161.985,67161.985538,6716198,6716199,6716198,0.01,0.109735,...,0.0,67161.985,0.000000,67161.985,0.000000,NaN,NaN,0.000000,0.000000,0.0


In [5]:
# STEP 5 — ALIGN BOTH DATASETS

# Now regime + outcome are aligned.

final = regime_df.merge(
    eval_df[["future_return", "future_volatility", "future_direction"]],
    left_index=True,
    right_index=True,
    how="inner"
)

# STEP 6 — ANALYZE REGIMES
regime_outcomes = final.groupby("regime").agg({
    "future_return": "mean",
    "future_volatility": "mean",
    "future_direction": "mean"
})

z = final.copy()

for col in feature_cols:
    z[col] = (z[col] - z[col].mean()) / z[col].std()

regime_profile = (
    z.groupby("regime")[feature_cols]
    .mean()
    .round(2)
)

full_profile = pd.DataFrame(regime_profile.join(regime_outcomes))
full_profile

,volatility,spread,order_imbalance,trade_imbalance,quote_churn,inventory,inventory_vol,microprice_error,future_return,future_volatility,future_direction
regime,,,,,,,,,,,
0,-0.13,-0.03,0.02,0.02,NaN,0.01,-0.14,-0.00,0.000002,0.000005,0.017926
1,0.58,-0.03,-0.12,-0.11,NaN,-0.03,0.80,0.02,-0.000002,0.000009,-0.018152
2,4.54,3.72,-0.02,-0.10,NaN,-0.21,1.18,-0.29,-0.000015,0.000026,-0.085252


In [6]:
def export_gmm(model_name, gmm, scaler, horizon_ms, feature_cols, regime_labels):
    K = gmm.n_components

    means = gmm.means_

    covs = gmm.covariances_
    precisions = gmm.precisions_  # inverse covariance (what you want)

    log_weights = np.log(gmm.weights_)

    # log determinant of covariance
    log_det = np.array([
        np.log(np.linalg.det(covs[k]))
        for k in range(K)
    ])

    artifact = {
        # GMM
        "means": means.tolist(),
        "cov_inv": precisions.tolist(),
        "log_det_cov": log_det.tolist(),
        "log_weights": log_weights.tolist(),

        # scaler (CRITICAL)
        "scaler_mean": scaler.mean_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),

        # metadata
        "model_name": model_name,
        "target": "detect_regime",
        "n_regimes": K,
        "horizon_ms": horizon_ms,
        "feature_cols": feature_cols,
        "regime_labels": [
            regime_labels[i] for i in range(K)
        ]
    }

    with open(f"data/{model_name}.json", "w") as f:
        json.dump(artifact, f)

    print(f"['data/{model_name}.json']")

In [7]:
regime_labels = {
    0: "low_vol",
    1: "toxic",
    2: "high_vol",
}

export_gmm(model_name="regime_model_5",
           gmm=model,
           scaler=scaler,
           horizon_ms=horizon_ms,
           feature_cols=feature_cols,
           regime_labels=regime_labels)

artifact = {
    "scaler": scaler,
    "model": model,
    "feature_cols": feature_cols,
    "n_regimes": n_regimes,
    "window": WINDOW,
    "horizon_ms": 1000,
    "regime_labels": regime_labels
}

joblib.dump(artifact, "data/regime_model_5.pkl")

['data/regime_model_5.json']


['data/regime_model_5.pkl']